# Test resources: pancreas

Generates the `pancreas` test resources with the `process_openproblems_v1` workflow and registers them in the public LaminDB instance [`openproblems/data-dev`](https://lamin.ai/openproblems/data-dev): one artifact per zarr store, grouped in the collection `resources_test/common/pancreas`.

The pipeline is deterministic, so rerunning this notebook without changes returns the existing artifacts and collection. Any change to the outputs creates a new version of the collection.

Requirements: Docker, Nextflow, components built with `viash ns build --setup cachedbuild`, and write access to `openproblems/data-dev`. LaminDB tracks a notebook run by `nbconvert` only with `--inplace`, so run a copy:

```bash
tmp=$(mktemp -d) && cp scripts/create_test_resources/pancreas.ipynb $tmp/
REPO_ROOT=$(pwd) jupyter nbconvert --to notebook --execute --inplace $tmp/pancreas.ipynb
```

In [ ]:
import os
import subprocess
import tempfile
import urllib.request
from pathlib import Path

import lamindb as ln

instance = "openproblems/data-dev"
ln.connect(instance)
# lamindb can silently fall back to the default instance, so check before writing anything
connected = ln.setup.settings.instance.slug
assert connected == instance, f"Connected to '{connected}' instead of '{instance}'"

ln.track()

In [ ]:
# REPO_ROOT is needed when running a copy of this notebook outside the repository
repo_root = Path(os.environ.get("REPO_ROOT") or subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True).stdout.strip())
key_prefix = "resources_test/common/pancreas"
output_dir = Path(tempfile.mkdtemp())

## Run the pipeline

Subsample the scIB pancreas dataset to 600 cells and 1500 genes, keeping the acinar and beta cells of three technologies and the cell cycle genes used by scIB.

In [ ]:
scib_url = "https://raw.githubusercontent.com/theislab/scib/c993ffd9ccc84ae0b1681928722ed21985fb91d1/scib/resources"
keep_features = []
for name in ["g2m_genes_tirosh_hm.txt", "s_genes_tirosh_hm.txt"]:
    with urllib.request.urlopen(f"{scib_url}/{name}") as response:
        keep_features += response.read().decode().split()

In [ ]:
# the knn output is the same store as dataset, so it is not published separately
outputs = ["raw", "normalized", "hvg", "pca", "dataset"]

subprocess.run(
    [
        "nextflow", "run", ".",
        "-main-script", "target/nextflow/workflows/scrnaseq/process_openproblems_v1/main.nf",
        "-profile", "docker",
        "-c", "common/nextflow_helpers/labels_ci.config",
        "--id", "pancreas",
        "--input_id", "pancreas",
        "--obs_cell_type", "celltype",
        "--obs_batch", "tech",
        "--var_feature_name", "index",
        "--layer_counts", "counts",
        "--dataset_name", "Human pancreas",
        "--dataset_url", "https://theislab.github.io/scib-reproducibility/dataset_pancreas.html",
        "--dataset_reference", "luecken2022benchmarking",
        "--dataset_summary", "Human pancreas cells dataset from the scIB benchmarks",
        "--dataset_description", "Human pancreatic islet scRNA-seq data from 6 datasets across technologies (CEL-seq, CEL-seq2, Smart-seq2, inDrop, Fluidigm C1, and SMARTER-seq).",
        "--dataset_organism", "homo_sapiens",
        "--keep_cell_type_categories", "acinar;beta",
        "--keep_batch_categories", "celseq;inDrop4;smarter",
        "--keep_features", ";".join(keep_features),
        "--seed", "123",
        "--normalization_methods", "log_cp10k",
        "--do_subsample", "true",
        "--n_obs", "600",
        "--n_vars", "1500",
        *[arg for name in outputs for arg in (f"--output_{name}", f"$id/{name}.anndata.zarr")],
        "--output_state", "$id/state.yaml",
        "--publish_dir", str(output_dir),
    ],
    cwd=repo_root,
    env={**os.environ, "NXF_SYNTAX_PARSER": "v1"},
    check=True,
)

## Register in LaminDB

`overwrite_versions=False` keeps the files of earlier versions, so a pinned version of the collection stays loadable.

In [ ]:
artifacts = [
    ln.Artifact(
        output_dir / "pancreas" / f"{name}.anndata.zarr",
        key=f"{key_prefix}/{name}.anndata.zarr",
        description=f"Test resource: pancreas, {name}",
        overwrite_versions=False,
    ).save()
    for name in outputs
]

In [ ]:
collection = ln.Collection(
    artifacts,
    key=key_prefix,
    description="Test resources: human pancreas from the scIB benchmarks, subsampled to 600 cells and 1500 genes",
).save()
collection.describe()
print(f"lamin://{instance}/collection/{collection.uid}")

In [ ]:
ln.finish()